In [ ]:
# Imports, Configre Pandas
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
# Reduce Data and Count initial Rows
engine = create_engine('postgresql://markfinlay:@localhost:5432/osm')

# Execute a DELETE query to reduce the dataset down
delete_query = """
DELETE FROM planet_osm_line
WHERE NOT (
    (highway IN ('path', 'footway', 'track') OR route = 'hiking')
    AND way IS NOT NULL
    AND NOT ST_IsEmpty(way)
    AND ST_Length(way) > 1000
    AND surface IS NOT NULL
);
"""

## Why am I limiting it to 1KM?

with engine.begin() as conn:
    conn.execute(text(delete_query))

print("Irrelevant rows have been deleted.")

query = """
    SELECT name, surface, width, tags, way, ST_Length(way) AS length_m FROM planet_osm_line
    WHERE (highway IN ('path', 'footway', 'track') OR route = 'hiking')
    AND way IS NOT NULL
    AND NOT ST_IsEmpty(way)
    AND ST_Length(way) > 1000
    AND surface IS NOT NULL
    ORDER BY osm_id;;"""

# Test the connection by reading a small sample from the table
df = pd.read_sql(query, engine)
df.count()

# TODO:: Map tags to individual columms

In [ ]:
df.head()

In [ ]:
df['length_m'].describe()

In [ ]:
df['surface'].value_counts()

In [ ]:
# PReview Data on a map
import geopandas as gpd

gdf = gpd.read_postgis(query, engine, geom_col='way')
# Convert to WGS84 so that the GeoJSON is in longitude/latitude
gdf = gdf.to_crs(epsg=4326)
gdf.explore()

In [ ]:
# Have a close look at Wikclow Hills Data 
# ST_MakeEnvelope(<min_lon>, <min_lat>, <max_lon>, <max_lat>, 4326)
query = """
    SELECT osm_id, name, highway, surface, way, ST_Length(way) AS length_m
    FROM planet_osm_line
    WHERE (highway IN ('path', 'footway') OR route = 'hiking')
    AND ST_Length(way) > 1000
    AND surface IS NOT NULL
    AND ST_Intersects( way, ST_Transform(ST_MakeEnvelope(-6.50, 52.9, -6.2, 53.24, 4326), 3857));
    """

gdf = gpd.read_postgis(query, engine, geom_col='way')
# Convert to WGS84 so that the GeoJSON is in longitude/latitude
gdf = gdf.to_crs(epsg=4326)
gdf.explore(zoom_start=11, location=(53.1, -6.3))

In [ ]:
# 2. Create the 'surface_group' column and populate it
from sqlalchemy import create_engine, text

with engine.begin() as conn:
    # a) Add the column if it doesn’t already exist
    conn.execute(text("""
        ALTER TABLE planet_osm_line
        ADD COLUMN IF NOT EXISTS surface_group text;
    """))

    # b) Update the column using a CASE expression
    conn.execute(text("""
        UPDATE planet_osm_line
        SET surface_group = CASE
            WHEN lower(surface) IN (
                'asphalt', 'asphaltsidewalk', 'asphalt;paved', 
                'tarmac', 'bituminous', 'macadam'
            )
            THEN 'asphalt'
            WHEN lower(surface) IN (
                 'concrete', 'concrete asphalt', 'concrete:plates',
                 'concrete:lanes', 'concrete_slabs', 'concrete_with_bitumen', 'cement',
                'unpaved/paved', 'paved', 'paving_stones', 'paving_stones:30',
                'brick', 'bricks', 'paving_stones', 'chipseal',
                'cobblestone', 'cobblestone:flattened'
            )
            THEN 'paved'

            WHEN lower(surface) IN (
                 'gravel', 'fine_gravel', 'gravel,_dirt', 'gravel_and_rock', 'aggregate',
                 'sand/gravel/stones', 'ground;gravel', 'compacted',  'sand;pebbles'
               )
            THEN 'gravel'

            WHEN lower(surface) IN (
                 'dirt', 'dust', 'earth', 'earth,_grass', 'dirt/sand',
                 'dirt; gravel; dirt', 'unpaved', 
                 'ground', 'clay', 'soil'
               )
            THEN 'mud/dirt'

            WHEN lower(surface) IN (
                 'grass', 'grass/trees', 'grass_paver', 'green'
               )
            THEN 'grass'

            ELSE NULL
        END
        WHERE surface IS NOT NULL;
    """))

In [ ]:
# Count instances per surface_group
df_surface_groups = pd.read_sql("""
    SELECT surface_group, COUNT(*) AS count
    FROM planet_osm_line
    WHERE (highway IN ('path', 'footway') OR route = 'hiking')
    AND ST_Length(way) > 1000
    AND surface IS NOT NULL
    GROUP BY surface_group
    ORDER BY count DESC
""", engine)
df_surface_groups

In [ ]:
# Simple bar chart of these frequencies
query = """
    SELECT osm_id, highway, surface, surface_group, tags, way, ST_Length(way) AS length_m
    FROM planet_osm_line
    WHERE (highway IN ('path', 'footway') OR route = 'hiking')
    AND ST_Length(way) > 1000
    AND surface IS NOT NULL
    AND surface_group IS NOT NULL
    """

gdf = gpd.read_postgis(query, engine, geom_col='way')
print(gdf.count())

gdf['surface_group'].value_counts().plot.bar(figsize=(8, 4))

In [ ]:
# Balance the data so that we have an even count matching the smallest class
import pandas as pd

# Define your target count per class
target_count = 1843 # 

# Create a list to collect each balanced group
balanced_frames = []

# Sample randomly by proportionality???

# Iterate over each surface_group
for group, group_data in gdf.groupby('surface_group'):
    if len(group_data) >= target_count:
        # Downsample majority groups
        balanced_group = group_data.sample(n=target_count, random_state=42)
    else:
        # Oversample minority groups (with replacement) to reach target_count
        balanced_group = group_data.sample(n=target_count, replace=True, random_state=42)
    balanced_frames.append(balanced_group)

# Combine all groups back into a single GeoDataFrame
gdf_balanced = pd.concat(balanced_frames, ignore_index=True)

# Verify the result
print(gdf_balanced['surface_group'].value_counts())

In [ ]:
# Display the interactive map of balanced data
# Ensure the CRS is set to WGS84 (longitude/latitude) for proper mapping
gdf_balanced = gdf_balanced.to_crs(epsg=3035)

map_balanced = gdf_balanced.explore(
    tooltip='surface_group',  # Show the surface_group when hovering
    popup=True,               # Enable popup on click (shows all attributes)
    tiles='OpenStreetMap'     # You can change the base map style if desired
)

map_balanced

In [ ]:
gdf_balanced.head(10)

In [ ]:
# Write the balanced GeoDataFrame to a new table in the database.
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine

gdf_balanced = gdf_balanced.to_crs(epsg=3035)

gdf_balanced.to_postgis('balanced_planet_osm_line', engine, if_exists='replace', index=False)
print("New table 'balanced_planet_osm_line' has been created and the balanced data is stored.")

In [ ]:
# Select all rows from the balanced table
query = "SELECT * FROM balanced_planet_osm_line;"

# Load the data into a GeoDataFrame.
# Adjust the 'geom_col' parameter if your geometry column is named differently.
balanced_gdf = gpd.read_postgis(query, engine, geom_col='way')

# Display the first few rows
balanced_gdf.head()

In [ ]:
# Descriptive statistics for numerical columns (e.g., length_m, width if numeric)
print("Descriptive statistics for numerical columns:")
balanced_gdf['length_m'].describe()

In [ ]:
# Count of each class in the 'surface_group' column
print("\nDistribution of 'surface_group':")
balanced_gdf['surface_group'].value_counts()

In [ ]:
# group-wise statistics for a specific column, you can use groupby.
# For example, the mean length_m per surface_group:
print("\nMean 'length_m' per surface_group:")
balanced_gdf.groupby('surface_group')['length_m'].mean()

In [ ]:
# Process each trail from the GeoDataFrame
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from shapely.geometry import LineString
import geopandas as gpd
from sentinelhub import CRS, BBox

# -------------------------------
# Parameters
# -------------------------------
segment_length = 320  # length in meters for each segment
fixed_width = 320     # width of bounding box (meters)
fixed_height = 320    # height of bounding box (meters)
half_width = fixed_width / 2
half_height = fixed_height / 2

# -------------------------------
# Read Balanced Trails
# -------------------------------
# Query the balanced table
balanced_query = "SELECT osm_id, surface_group, way, ST_Length(way) AS length_m FROM balanced_planet_osm_line;"
# Load as GeoDataFrame (assuming the geometry column is named "way")
balanced_gdf = gpd.read_postgis(balanced_query, engine, geom_col='way')

# For segmentation in metric units, ensure geometries are in an appropriate meter-based CRS
# EPSG:3035 (ETRS89 / LAEA Europe) is an equal-area projection for Europe, including the UK and Ireland, 
# in a single coordinate system. It’s often used as a compromise projection for multi-country datasets
balanced_gdf = balanced_gdf.to_crs(epsg=3035)

print(f"Read {len(balanced_gdf)} trails from balanced_planet_osm_line.")

# -------------------------------
# Create the trail_segments Table
# -------------------------------
create_table_sql = """
DROP TABLE IF EXISTS trail_segments;
CREATE TABLE trail_segments (
    id SERIAL PRIMARY KEY,
    filename TEXT,
    osm_id BIGINT,
    surface_group TEXT,
    center_geom geometry(Point,3035),
    image_data BYTEA
);
"""
with engine.begin() as conn:
    conn.execute(text(create_table_sql))
print("New table 'trail_segments' has been created.")

# -------------------------------
# Segment Each Trail and Insert Rows
# -------------------------------
# Define an SQL insert statement.
# We use PostGIS functions ST_SetSRID(ST_MakePoint(...)) for the center point
# and ST_GeomFromText() for the bounding box (using WKT).
insert_sql = """
INSERT INTO trail_segments (filename, osm_id, surface_group, center_geom, image_data)
VALUES (
    :filename,
    :osm_id,
    :surface_group,
    ST_SetSRID(ST_MakePoint(:center_lon, :center_lat), 3035),
    :image_data
);
"""

for idx_row, trail_row in balanced_gdf.iterrows():
    osm_id = trail_row['osm_id']
    trail_metric = trail_row['way'] 
    
    # --- Segment the Trail ---
    num_segments = int(trail_metric.length // segment_length)
    segments = []
    for i in range(num_segments):
        start = trail_metric.interpolate(i * segment_length)
        end = trail_metric.interpolate((i + 1) * segment_length)
        segments.append(LineString([start, end]))
    # Add remaining piece if necessary
    if trail_metric.length % segment_length > 0:
        segments.append(LineString([trail_metric.interpolate(num_segments * segment_length),
                                     trail_metric.coords[-1]]))
    
    # Process each segment.
    for idx_seg, seg in enumerate(segments):
        # Compute the center point of the segment.
        center = seg.centroid
        center_lon, center_lat = center.x, center.y
        # Construct a filename for this segment.
        filename = f"{osm_id}_segment_{idx_seg + 1}"
    
        # Prepare parameters for insertion.
        params = {
            "filename": filename,
            "osm_id": int(osm_id),
            "surface_group": trail_row['surface_group'],
            "center_lon": center_lon,
            "center_lat": center_lat,
            "image_data": None  # placeholder (NULL); you can later update this field.
        }
    
        # Insert the segment row into the trail_segments table.
        with engine.begin() as conn:
            conn.execute(text(insert_sql), params)

print("All trail segments have been processed and stored in 'trail_segments'.")

In [ ]:
# Select all rows from trail_segments
query = "SELECT * FROM trail_segments;"

# Load the data into a GeoDataFrame.
trail_segments_gpdf= gpd.read_postgis(query, engine, geom_col='center_geom')

# Display the first few rows
trail_segments_gpdf.head()

In [ ]:
from matplotlib.patches import Rectangle

# Process each trail from the GeoDataFrame
sampled_trails = balanced_gdf.sample(n=20, random_state=42)  
for idx_row, trail_row in sampled_trails.iterrows():
    osm_id = trail_row['osm_id']
    trail_metric = trail_row['way']
    
    # --- Visualization ---
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot the original trail in light gray
    x, y = trail_metric.xy
    ax.plot(x, y, color='lightgray', linewidth=2, label='Original Trail')

    # Select all segments by osm_id and load them into a GeoDataFrame
    query = f"SELECT * FROM trail_segments WHERE osm_id = {osm_id}"
    trail_segments_gpdf = gpd.read_postgis(query, engine, geom_col='center_geom').to_crs(epsg=3035)

    # Plot the centroids from the trail segments
    ax.scatter(trail_segments_gpdf['center_geom'].x, 
               trail_segments_gpdf['center_geom'].y, 
               color='red', s=50, label='Trail Segment Centroid')
    
    # For each centroid, add a 320m x 320m bounding box centered on it
    for idx, row in trail_segments_gpdf.iterrows():
        centroid = row['center_geom']
        lower_left_x = centroid.x - 160  # 320/2
        lower_left_y = centroid.y - 160  # 320/2
        rect = Rectangle((lower_left_x, lower_left_y), 320, 320, 
                         linewidth=1, edgecolor='blue', facecolor='none')
        ax.add_patch(rect)
    
    ax.set_aspect('equal', 'box')
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.set_title(f"Trail with 320m x 320m Bounding Boxes (OSM id: {osm_id})")
    ax.legend()
    plt.show()
